# Crypto Zombies — classroom demo (CSV only)

**Paper:** Będowska-Sójka, B., Wójcik, P., & Pele, D. T. (2025). *Early warning systems for cryptocurrency markets: Predicting ‘zombie’ assets using machine learning.* *The North American Journal of Economics and Finance*, 102543. DOI [10.1016/j.najef.2025.102543](https://doi.org/10.1016/j.najef.2025.102543)

**Scope for class:** we use **only** the unpacked QuantLet file

`experiments/Crypto_Zombies/workdir/_cryptos_weekly_snapshots/all_symbols.csv`

No aux xlsx, no Dropbox helpers, no full ROSE/SMOTE horse race from `Code/`.
This is a **faithful sketch of the research design** (label, OOT split, trees, importance), not a full paper replication.

**Companion:** `sessions/05_crypto_zombies.md` · `experiments/crypto_zombies_runbook.txt`

## 0. Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

DATA = Path("experiments/Crypto_Zombies/workdir/_cryptos_weekly_snapshots/all_symbols.csv")
if not DATA.is_file():
    raise FileNotFoundError(
        "Crypto data not found. These data (about 1 GB) are not part of the package. experiments/crypto_zombies_runbook.txt says how to get them from QuantLet and where to put the file."
    ) from None

rng = np.random.default_rng(2025)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Load & define the early-warning task

Label (paper spirit): `delisted_in_next28days` — asset becomes a “zombie” / untradable within 28 days.

We **exclude** other horizon labels from features (leakage).

In [ ]:
# ~860k rows / ~1GB — load once; keep needed columns
usecols = None  # full file; filter columns after
raw = pd.read_csv(DATA, low_memory=False)
raw["date"] = pd.to_datetime(raw["date"])
print(raw.shape)
print("date range:", raw["date"].min().date(), "→", raw["date"].max().date())
print("P(zombie in 28d) =", round(raw["delisted_in_next28days"].mean(), 4))

In [ ]:
leak_cols = [
    "delisted_in_next7days", "delisted_in_next14days",
    "delisted_in_next21days", "delisted_in_next28days",
]
id_cols = ["id", "symbol", "slug", "date", "weekday"]
# optional raw levels often heavy-tailed — keep roll_* stats as main signal
drop_extra = ["close", "volume", "market_cap", "market_cap_vol_close", "r", "is_active", "life"]

feature_cols = [
    c for c in raw.columns
    if c not in leak_cols + id_cols + drop_extra
]
print(len(feature_cols), "features")
print(feature_cols[:12], "...")

df = raw[["date", "delisted_in_next28days"] + feature_cols].copy()
df = df.rename(columns={"delisted_in_next28days": "y"})
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print("after dropna:", df.shape, "| P(y=1)=", round(df["y"].mean(), 4))

## 2. Out-of-time split (one window)

Classroom single cut (paper uses a walk-forward loop).  
Train: through 2021-06-30 · Test: 2021-07-01 → 2021-12-31.

In [ ]:
train_end = pd.Timestamp("2021-06-30")
test_end = pd.Timestamp("2021-12-31")

train = df[df["date"] <= train_end]
test = df[(df["date"] > train_end) & (df["date"] <= test_end)]

# speed: optional downsample majority class in train only
def balance_train(d, max_neg_per_pos=5, seed=2025):
    pos = d[d["y"] == 1]
    neg = d[d["y"] == 0]
    n_neg = min(len(neg), max_neg_per_pos * max(len(pos), 1))
    neg_s = neg.sample(n=n_neg, random_state=seed)
    return pd.concat([pos, neg_s], ignore_index=True).sample(frac=1.0, random_state=seed)

train_fit = balance_train(train)
print("train full:", train.shape, "P=", round(train["y"].mean(), 4))
print("train fit :", train_fit.shape, "P=", round(train_fit["y"].mean(), 4))
print("test      :", test.shape, "P=", round(test["y"].mean(), 4))

X_train = train_fit[feature_cols].to_numpy()
y_train = train_fit["y"].to_numpy()
X_test = test[feature_cols].to_numpy()
y_test = test["y"].to_numpy()

## 3. Model ladder: Logit vs Random Forest

Matches the paper’s qualitative message: trees beat a simple linear baseline on this task.

In [ ]:
models = {
    "Logit": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, class_weight="balanced")),
    ]),
    "RF": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced_subsample",
        random_state=2025,
        n_jobs=-1,
    ),
}

rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "model": name,
        "balanced_acc": balanced_accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba),
        "ap": average_precision_score(y_test, proba),
    })
    print(name)
    print(classification_report(y_test, pred, digits=3))

board = pd.DataFrame(rows).sort_values("balanced_acc", ascending=False)
board.round(3)

## 4. Threshold / cost discussion

False negative = investor stuck in a zombie.  
False positive = unnecessary exit. Re-rank by a simple cost matrix.

In [ ]:
def expected_cost(y_true, proba, tau, c_fp=1.0, c_fn=5.0):
    y_hat = (proba >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return (c_fp * fp + c_fn * fn) / len(y_true), dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

best = board.iloc[0]["model"]
proba = fitted[best].predict_proba(X_test)[:, 1]
cost_rows = []
for tau in np.linspace(0.2, 0.8, 13):
    c, cm = expected_cost(y_test, proba, tau)
    cost_rows.append({"tau": tau, "exp_cost": c, **cm})
cost_df = pd.DataFrame(cost_rows)
print("best model on board:", best)
cost_df.loc[cost_df["exp_cost"].idxmin()]

## 5. Lightweight XAI — RF impurity importance

Stand-in for the paper’s DALEX / PDP stack (needs the full R pipeline).

In [ ]:
rf = fitted["RF"]
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
top = imp.head(15)
ax = top.iloc[::-1].plot(kind="barh", title="RF feature importance (top 15)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()
top.round(4)

## 6. Discussion questions

1. Why must we drop `delisted_in_next7/14/21days` from \(X\)?  
2. How does majority downsampling in train change what “accuracy” means?  
3. Map volumes / return stats in the top-15 to the paper’s narrative.  
4. What is still missing vs the published QuantLet scripts (tags, ROSE/SMOTE grid, full walk-forward)?

## Takeaway

With **only** the public CSV we can still teach: label → OOT split → baseline vs RF → costs → importance.  
Full replication stays optional homework when aux files are available.